# Quantification / Visualization Analysis Driver

목적: pipeline 실행 후 산출물을 불러와 정량화, 시각화, 분석합니다. LIF-to-TIFF 단계의 MIP, histogram, Fourier QC, Z-stack GIF 시각화/저장 코드를 TIFF stack 기반으로 모아둡니다.


## 0. Setup


In [ ]:
from __future__ import annotations

import base64
import json
from io import BytesIO
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile
from PIL import Image as PILImage
from IPython.display import Image, HTML, display
from matplotlib.colors import LinearSegmentedColormap
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
RESULTS_DIR = PROJECT_DIR / "analysis_results"
QC_DIR = RESULTS_DIR / "lif_to_tiff_qc"
GIF_DIR = QC_DIR / "z_stack_gifs"
MIP_DIR = QC_DIR / "mip_panels"
HIST_FFT_DIR = QC_DIR / "histogram_fourier"
for path in [RESULTS_DIR, QC_DIR, GIF_DIR, MIP_DIR, HIST_FFT_DIR]:
    path.mkdir(exist_ok=True, parents=True)

CHANNELS = ("DAPI", "Reference", "Target")
CMAP_BLUE = LinearSegmentedColormap.from_list("dapi", ["black", "blue"])
CMAP_GREEN = LinearSegmentedColormap.from_list("reference", ["black", "green"])
CMAP_RED = LinearSegmentedColormap.from_list("target", ["black", "red"])
CHANNEL_CMAPS = {"DAPI": CMAP_BLUE, "Reference": CMAP_GREEN, "Target": CMAP_RED}

# Test-set controls for fast iteration.
TEST_MODE = True
SELECT_LIF_NAMES = None        # e.g. ["1_PSD Ms MA1046_Homer Rb SYSY"] or None
SELECT_SERIES_CONTAINS = "63x"  # substring filter; None disables
SELECT_SERIES_NAMES = None     # exact series-folder names; None disables
MAX_TEST_SERIES = 3 if TEST_MODE else None

# Visualization controls.
SELECT_INDEX = 0
SHOW_Z_GIF_INLINE = True
SAVE_QC_FIGURES = True
SAVE_Z_STACK_GIFS = True
GIF_DURATION_MS = 120
GIF_DISPLAY_WIDTH = 280
SCALE_BAR_UM = 1000

print({
    "project_dir": str(PROJECT_DIR),
    "data_dir": str(DATA_DIR),
    "results_dir": str(RESULTS_DIR),
    "test_mode": TEST_MODE,
    "max_test_series": MAX_TEST_SERIES,
    "save_qc_figures": SAVE_QC_FIGURES,
    "save_z_stack_gifs": SAVE_Z_STACK_GIFS,
})


## 1. Discover Series and Metadata


In [ ]:
def _series_passes_filter(lif_name: str, series_name: str):
    if SELECT_LIF_NAMES:
        allowed = set(SELECT_LIF_NAMES)
        if lif_name not in allowed and f"{lif_name}.lif" not in allowed:
            return False
    if SELECT_SERIES_NAMES and series_name not in set(SELECT_SERIES_NAMES):
        return False
    if SELECT_SERIES_CONTAINS and SELECT_SERIES_CONTAINS.lower() not in series_name.lower():
        return False
    return True


def apply_series_limit(df: pd.DataFrame):
    if df.empty:
        return df
    if MAX_TEST_SERIES is None:
        return df.reset_index(drop=True)
    return df.head(int(MAX_TEST_SERIES)).reset_index(drop=True)


def discover_series(data_dir: Path = DATA_DIR, apply_filter: bool = True):
    rows = []
    for stacks_dir in sorted(data_dir.glob("*/*/stacks")) if data_dir.exists() else []:
        series_dir = stacks_dir.parent
        lif_name = series_dir.parent.name
        series_name = series_dir.name
        if apply_filter and not _series_passes_filter(lif_name, series_name):
            continue
        reg_dir = series_dir / "registered_stacks"
        metadata_path = series_dir / "metadata.json"
        rows.append({
            "lif_name": lif_name,
            "series_name": series_name,
            "series_dir": series_dir,
            "stacks_dir": stacks_dir,
            "registered_stacks_dir": reg_dir,
            "has_raw_stacks": all((stacks_dir / f"{ch}_stack.tif").exists() for ch in CHANNELS),
            "has_registered_stacks": all((reg_dir / f"{ch}_stack_registered.tif").exists() for ch in CHANNELS),
            "metadata_path": metadata_path if metadata_path.exists() else None,
        })
    df = pd.DataFrame(rows)
    return apply_series_limit(df) if apply_filter else df.reset_index(drop=True)


all_series_df = discover_series(apply_filter=False)
series_df = discover_series(apply_filter=True)
print(f"All series: {len(all_series_df)} | selected/test series: {len(series_df)}")
display(series_df)


## 2. LIF-to-TIFF Visualization Helpers


In [ ]:
def sanitize_name(name):
    bad_chars = ['/', '\\', ':', '*', '?', '"', '<', '>', '|']
    out = str(name)
    for ch in bad_chars:
        out = out.replace(ch, "_")
    return out.strip()


def load_metadata(path):
    if path is None or not Path(path).exists():
        return {}
    return json.loads(Path(path).read_text(encoding="utf-8"))


def normalize_for_view(data, p_low=1, p_high=99.8):
    data = np.asarray(data)
    if data.size == 0 or np.all(data == data.flat[0]):
        return np.zeros_like(data, dtype=np.float32)
    lo, hi = np.percentile(data, (p_low, p_high))
    if hi <= lo:
        return np.zeros_like(data, dtype=np.float32)
    return np.clip((data.astype(np.float32) - lo) / (hi - lo), 0, 1).astype(np.float32)


def ensure_uint8(arr):
    arr = np.asarray(arr)
    if np.issubdtype(arr.dtype, np.uint8):
        return arr
    if np.issubdtype(arr.dtype, np.floating):
        arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    return np.clip(arr, 0, 255).astype(np.uint8)


def compute_fft_log_image(img):
    f = np.fft.fft2(img)
    fshift = np.fft.fftshift(f)
    mag = np.abs(fshift)
    return np.log1p(mag)


def get_scale_bar_length_pixels_from_metadata(metadata: dict, bar_length_um=SCALE_BAR_UM):
    scale = metadata.get("scale") or {}
    px_size = scale.get("x")
    try:
        px_size = float(px_size)
    except (TypeError, ValueError):
        return None
    if px_size <= 0:
        return None
    return float(bar_length_um) / px_size


def read_channel_stack(series_dir: Path, channel: str, registered: bool = False):
    series_dir = Path(series_dir)
    if registered:
        p = series_dir / "registered_stacks" / f"{channel}_stack_registered.tif"
    else:
        p = series_dir / "stacks" / f"{channel}_stack.tif"
    if not p.exists():
        raise FileNotFoundError(p)
    return tifffile.imread(str(p))


def load_channel_stacks(series_dir: Path, registered: bool = False, channels=CHANNELS):
    return {ch: read_channel_stack(series_dir, ch, registered=registered) for ch in channels}


def channel_data_from_stacks(stacks: dict[str, np.ndarray]):
    return {
        ch: {"stack": stack, "mip": np.max(stack, axis=0)}
        for ch, stack in stacks.items()
    }


def merged_mip_from_channel_data(channel_data, channels=CHANNELS):
    dapi = channel_data[channels[0]]["mip"]
    ref = channel_data[channels[1]]["mip"]
    target = channel_data[channels[2]]["mip"]
    merged = np.zeros((*dapi.shape, 3), dtype=np.float32)
    merged[..., 0] = normalize_for_view(target)
    merged[..., 1] = normalize_for_view(ref)
    merged[..., 2] = normalize_for_view(dapi)
    return merged


def stack_to_gif_bytes(stack, duration_ms=120, color=None):
    stack = np.asarray(stack)
    if stack.ndim != 3:
        raise ValueError(f"Expected stack shape (Z, Y, X), got {stack.shape}")
    frames = []
    for z_idx in range(stack.shape[0]):
        frame = ensure_uint8(np.round(normalize_for_view(stack[z_idx]) * 255.0))
        if color is None:
            frames.append(PILImage.fromarray(frame, mode="L"))
            continue
        rgb = np.zeros((*frame.shape, 3), dtype=np.uint8)
        if color == "red":
            rgb[..., 0] = frame
        elif color == "green":
            rgb[..., 1] = frame
        elif color == "blue":
            rgb[..., 2] = frame
        else:
            raise ValueError(f"Unsupported color: {color}")
        frames.append(PILImage.fromarray(rgb, mode="RGB"))
    buffer = BytesIO()
    frames[0].save(buffer, format="GIF", save_all=True, append_images=frames[1:], duration=duration_ms, loop=0, optimize=False)
    return buffer.getvalue()


def make_reference_target_merged_gif_bytes(stacks: dict[str, np.ndarray], duration_ms=120):
    ref_stack = np.asarray(stacks["Reference"])
    target_stack = np.asarray(stacks["Target"])
    if ref_stack.shape != target_stack.shape:
        raise ValueError(f"Reference/Target shape mismatch: {ref_stack.shape} vs {target_stack.shape}")
    merged_frames = []
    for z in range(ref_stack.shape[0]):
        ref_u8 = ensure_uint8(np.round(normalize_for_view(ref_stack[z]) * 255.0))
        target_u8 = ensure_uint8(np.round(normalize_for_view(target_stack[z]) * 255.0))
        rgb = np.zeros((*ref_u8.shape, 3), dtype=np.uint8)
        rgb[..., 0] = target_u8
        rgb[..., 1] = ref_u8
        merged_frames.append(PILImage.fromarray(rgb, mode="RGB"))
    buffer = BytesIO()
    merged_frames[0].save(buffer, format="GIF", save_all=True, append_images=merged_frames[1:], duration=duration_ms, loop=0, optimize=False)
    return buffer.getvalue()


def show_series_mip_panel(series_dir: Path, registered: bool = False, figsize=(18, 5), save_path: Path | None = None):
    series_dir = Path(series_dir)
    stacks = load_channel_stacks(series_dir, registered=registered)
    channel_data = channel_data_from_stacks(stacks)
    merged = merged_mip_from_channel_data(channel_data)
    metadata = load_metadata(series_dir / "metadata.json")
    scale_bar_len = get_scale_bar_length_pixels_from_metadata(metadata)
    fig, axes = plt.subplots(1, 4, figsize=figsize, facecolor="black")
    fig.suptitle(f"{series_dir.parent.name} | {series_dir.name} | {'registered' if registered else 'raw'}", color="white", fontsize=14, fontweight="bold", y=0.98)
    images = [normalize_for_view(channel_data[ch]["mip"]) for ch in CHANNELS] + [merged]
    cmaps = [CMAP_BLUE, CMAP_GREEN, CMAP_RED, None]
    titles = list(CHANNELS) + ["Merged"]
    for ax, image, cmap, title_text in zip(axes, images, cmaps, titles):
        ax.imshow(image, cmap=cmap)
        ax.set_title(title_text, color="white")
        ax.axis("off")
        if scale_bar_len is not None:
            ax.add_artist(AnchoredSizeBar(ax.transData, scale_bar_len, f"{SCALE_BAR_UM} µm", "lower right", pad=0.6, borderpad=0.8, sep=6, color="white", frameon=False, size_vertical=10, fontproperties={"size": 12}))
    plt.tight_layout()
    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(str(save_path), dpi=150, facecolor=fig.get_facecolor(), bbox_inches="tight")
    plt.show()
    return channel_data


def show_series_histogram_fft(series_dir: Path, registered: bool = False, figsize=(15, 12), save_path: Path | None = None):
    series_dir = Path(series_dir)
    stacks = load_channel_stacks(series_dir, registered=registered)
    channel_data = channel_data_from_stacks(stacks)
    fig, axes = plt.subplots(3, 3, figsize=figsize, facecolor="black")
    fig.suptitle(f"{series_dir.name} | {'registered' if registered else 'raw'} | MIP / Histogram / Fourier", color="white", fontsize=14)
    channel_to_cmap = {"DAPI": "Blues", "Reference": "Greens", "Target": "Reds"}
    for row_idx, ch in enumerate(CHANNELS):
        mip = channel_data[ch]["mip"]
        img_disp = normalize_for_view(mip)
        fft_disp = normalize_for_view(compute_fft_log_image(mip), p_low=0, p_high=99.9)
        axes[row_idx, 0].imshow(img_disp, cmap=channel_to_cmap[ch])
        axes[row_idx, 0].set_title(f"{ch} Image", color="white")
        axes[row_idx, 0].axis("off")
        axes[row_idx, 1].hist(mip.ravel(), bins=256, color="white")
        axes[row_idx, 1].set_title(f"{ch} Histogram", color="white")
        axes[row_idx, 1].set_facecolor("black")
        axes[row_idx, 1].tick_params(colors="white")
        for spine in axes[row_idx, 1].spines.values():
            spine.set_color("white")
        axes[row_idx, 2].imshow(fft_disp, cmap="gray", vmin=0, vmax=1)
        axes[row_idx, 2].set_title(f"{ch} Fourier Space", color="white")
        axes[row_idx, 2].axis("off")
    plt.tight_layout()
    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(str(save_path), dpi=150, facecolor=fig.get_facecolor(), bbox_inches="tight")
    plt.show()
    return channel_data


def save_z_stack_gif_triplet(series_dir: Path, registered: bool = False, duration_ms=120):
    series_dir = Path(series_dir)
    stacks = load_channel_stacks(series_dir, registered=registered)
    suffix = "registered" if registered else "raw"
    out_dir = GIF_DIR / sanitize_name(series_dir.parent.name) / sanitize_name(series_dir.name)
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = {
        "Reference": out_dir / f"{sanitize_name(series_dir.name)}_{suffix}_Reference_zstack.gif",
        "Target": out_dir / f"{sanitize_name(series_dir.name)}_{suffix}_Target_zstack.gif",
        "Merged": out_dir / f"{sanitize_name(series_dir.name)}_{suffix}_Reference_Target_merged_zstack.gif",
    }
    paths["Reference"].write_bytes(stack_to_gif_bytes(stacks["Reference"], duration_ms=duration_ms, color="green"))
    paths["Target"].write_bytes(stack_to_gif_bytes(stacks["Target"], duration_ms=duration_ms, color="red"))
    paths["Merged"].write_bytes(make_reference_target_merged_gif_bytes(stacks, duration_ms=duration_ms))
    return paths


def show_reference_target_z_gif_triplet(series_dir: Path, registered: bool = False, duration_ms=120, display_width=280, save_files=False):
    series_dir = Path(series_dir)
    stacks = load_channel_stacks(series_dir, registered=registered)
    ref_gif = base64.b64encode(stack_to_gif_bytes(stacks["Reference"], duration_ms=duration_ms, color="green")).decode("ascii")
    target_gif = base64.b64encode(stack_to_gif_bytes(stacks["Target"], duration_ms=duration_ms, color="red")).decode("ascii")
    merged_gif = base64.b64encode(make_reference_target_merged_gif_bytes(stacks, duration_ms=duration_ms)).decode("ascii")
    saved_paths = save_z_stack_gif_triplet(series_dir, registered=registered, duration_ms=duration_ms) if save_files else None
    image_style = f"width:{display_width}px;height:auto;display:block;"
    merged_style = f"width:{display_width * 2}px;max-width:100%;height:auto;display:block;"
    label = "registered" if registered else "raw"
    saved_text = ""
    if saved_paths:
        saved_text = "<div style='color:white;font-size:12px;margin-top:8px;'>Saved: " + " | ".join(str(p) for p in saved_paths.values()) + "</div>"
    html = f"""
    <div style='background:black;padding:12px;border-radius:8px;'>
      <div style='color:white;font-weight:bold;margin-bottom:10px;'>{series_dir.name} | {label} | Z-stack GIF</div>
      <div style='display:grid;grid-template-columns:repeat(2, max-content);gap:18px;align-items:start;'>
        <div><div style='color:white;text-align:center;margin-bottom:6px;'>Reference</div><img src='data:image/gif;base64,{ref_gif}' style='{image_style}' /></div>
        <div><div style='color:white;text-align:center;margin-bottom:6px;'>Target</div><img src='data:image/gif;base64,{target_gif}' style='{image_style}' /></div>
        <div style='grid-column:1 / span 2;'><div style='color:white;text-align:center;margin-bottom:6px;'>Merged</div><img src='data:image/gif;base64,{merged_gif}' style='{merged_style}' /></div>
      </div>
      {saved_text}
    </div>
    """
    display(HTML(html))
    return saved_paths


def raw_stack_intensity_summary(series_df: pd.DataFrame):
    rows = []
    for row in series_df.itertuples(index=False):
        series_dir = Path(row.series_dir)
        for ch in CHANNELS:
            try:
                stack = read_channel_stack(series_dir, ch, registered=False)
            except FileNotFoundError:
                continue
            nz = stack[stack > 0]
            base = nz if nz.size else stack.reshape(-1)
            rows.append({
                "lif_name": row.lif_name,
                "series_name": row.series_name,
                "channel": ch,
                "z_slices": stack.shape[0],
                "y_size": stack.shape[1],
                "x_size": stack.shape[2],
                "nonzero_fraction": float(np.count_nonzero(stack) / stack.size),
                "mean": float(np.mean(base)),
                "median": float(np.median(base)),
                "p99_8": float(np.percentile(base, 99.8)),
                "max": float(np.max(stack)),
            })
    return pd.DataFrame(rows)


## 3. Metadata Summary


In [ ]:
metadata_rows = []
for row in series_df.itertuples(index=False):
    meta = load_metadata(row.metadata_path)
    reg = meta.get("registration") or meta.get("elastix_registration") or {}
    prep = meta.get("preprocessing") or {}
    metadata_rows.append({
        "lif_name": row.lif_name,
        "series_name": row.series_name,
        "z_slices": meta.get("z_slices"),
        "x_size": meta.get("x_size"),
        "y_size": meta.get("y_size"),
        "scale_x": (meta.get("scale") or {}).get("x"),
        "scale_z": (meta.get("scale") or {}).get("z"),
        "registration_method": reg.get("method"),
        "final_z_slices": reg.get("final_z_slices"),
        "clipped": reg.get("clipped_due_to_failure", reg.get("clipped_at") is not None if reg else None),
        "denoise_lines": prep.get("denoise_lines"),
    })

metadata_df = pd.DataFrame(metadata_rows)
display(metadata_df)


## 4. Review One Series

선택된 test series 하나를 대상으로 MIP panel, histogram/Fourier, Z-stack GIF를 확인합니다.


In [ ]:
if not series_df.empty:
    selected_series_dir = Path(series_df.iloc[SELECT_INDEX]["series_dir"])
    print(selected_series_dir)
    prefix = f"{sanitize_name(selected_series_dir.parent.name)}__{sanitize_name(selected_series_dir.name)}"
    mip_path = MIP_DIR / f"{prefix}__raw_mip_panel.png" if SAVE_QC_FIGURES else None
    hist_fft_path = HIST_FFT_DIR / f"{prefix}__raw_hist_fft.png" if SAVE_QC_FIGURES else None
    show_series_mip_panel(selected_series_dir, registered=False, save_path=mip_path)
    show_series_histogram_fft(selected_series_dir, registered=False, save_path=hist_fft_path)
    if SHOW_Z_GIF_INLINE:
        show_reference_target_z_gif_triplet(
            selected_series_dir,
            registered=False,
            duration_ms=GIF_DURATION_MS,
            display_width=GIF_DISPLAY_WIDTH,
            save_files=SAVE_Z_STACK_GIFS,
        )
    if (selected_series_dir / "registered_stacks").exists():
        reg_mip_path = MIP_DIR / f"{prefix}__registered_mip_panel.png" if SAVE_QC_FIGURES else None
        reg_hist_fft_path = HIST_FFT_DIR / f"{prefix}__registered_hist_fft.png" if SAVE_QC_FIGURES else None
        show_series_mip_panel(selected_series_dir, registered=True, save_path=reg_mip_path)
        show_series_histogram_fft(selected_series_dir, registered=True, save_path=reg_hist_fft_path)
        if SHOW_Z_GIF_INLINE:
            show_reference_target_z_gif_triplet(
                selected_series_dir,
                registered=True,
                duration_ms=GIF_DURATION_MS,
                display_width=GIF_DISPLAY_WIDTH,
                save_files=SAVE_Z_STACK_GIFS,
            )


## 5. Batch LIF-to-TIFF QC Export

선택된 test set 또는 전체 series에 대해 MIP panel, histogram/Fourier figure, Reference/Target/Merged Z-stack GIF 파일을 저장합니다.


In [ ]:
RUN_BATCH_QC_EXPORT = False
EXPORT_REGISTERED_TOO = False

qc_export_rows = []
if RUN_BATCH_QC_EXPORT:
    for row in series_df.itertuples(index=False):
        series_dir = Path(row.series_dir)
        prefix = f"{sanitize_name(row.lif_name)}__{sanitize_name(row.series_name)}"
        mip_path = MIP_DIR / f"{prefix}__raw_mip_panel.png"
        hist_fft_path = HIST_FFT_DIR / f"{prefix}__raw_hist_fft.png"
        show_series_mip_panel(series_dir, registered=False, save_path=mip_path)
        show_series_histogram_fft(series_dir, registered=False, save_path=hist_fft_path)
        gif_paths = save_z_stack_gif_triplet(series_dir, registered=False, duration_ms=GIF_DURATION_MS) if SAVE_Z_STACK_GIFS else {}
        qc_export_rows.append({
            "lif_name": row.lif_name,
            "series_name": row.series_name,
            "kind": "raw",
            "mip_panel": str(mip_path),
            "hist_fft": str(hist_fft_path),
            "reference_gif": str(gif_paths.get("Reference", "")),
            "target_gif": str(gif_paths.get("Target", "")),
            "merged_gif": str(gif_paths.get("Merged", "")),
        })
        if EXPORT_REGISTERED_TOO and (series_dir / "registered_stacks").exists():
            reg_mip_path = MIP_DIR / f"{prefix}__registered_mip_panel.png"
            reg_hist_fft_path = HIST_FFT_DIR / f"{prefix}__registered_hist_fft.png"
            show_series_mip_panel(series_dir, registered=True, save_path=reg_mip_path)
            show_series_histogram_fft(series_dir, registered=True, save_path=reg_hist_fft_path)
            reg_gif_paths = save_z_stack_gif_triplet(series_dir, registered=True, duration_ms=GIF_DURATION_MS) if SAVE_Z_STACK_GIFS else {}
            qc_export_rows.append({
                "lif_name": row.lif_name,
                "series_name": row.series_name,
                "kind": "registered",
                "mip_panel": str(reg_mip_path),
                "hist_fft": str(reg_hist_fft_path),
                "reference_gif": str(reg_gif_paths.get("Reference", "")),
                "target_gif": str(reg_gif_paths.get("Target", "")),
                "merged_gif": str(reg_gif_paths.get("Merged", "")),
            })
else:
    print("[SKIP] Set RUN_BATCH_QC_EXPORT = True to export QC files for selected/test series.")

qc_export_df = pd.DataFrame(qc_export_rows)
if not qc_export_df.empty:
    qc_export_df.to_csv(QC_DIR / "lif_to_tiff_qc_exports.csv", index=False)
    display(qc_export_df)


## 6. LIF-to-TIFF Raw Stack Intensity Summary


In [ ]:
raw_intensity_df = raw_stack_intensity_summary(series_df)
display(raw_intensity_df)
if not raw_intensity_df.empty:
    raw_intensity_df.to_csv(RESULTS_DIR / "lif_to_tiff_raw_stack_intensity_summary.csv", index=False)
    fig, ax = plt.subplots(figsize=(10, 4))
    raw_intensity_df.boxplot(column="p99_8", by="channel", ax=ax)
    ax.set_title("Raw TIFF stack p99.8 intensity by channel")
    ax.set_xlabel("Channel")
    ax.set_ylabel("p99.8 intensity")
    plt.suptitle("")
    plt.tight_layout()
    plt.show()


## 7. Registration Drift Summary


In [ ]:
drift_rows = []
for row in series_df.itertuples(index=False):
    meta = load_metadata(row.metadata_path)
    reg = meta.get("registration") or meta.get("elastix_registration") or {}
    log = reg.get("transform_log", [])
    if not log:
        continue
    tx = np.array([e.get("tx", 0.0) for e in log], dtype=float)
    ty = np.array([e.get("ty", 0.0) for e in log], dtype=float)
    reasons = [e.get("reason", "unknown") for e in log]
    drift_rows.append({
        "lif_name": row.lif_name,
        "series_name": row.series_name,
        "n_log_entries": len(log),
        "max_displacement": float(np.max(np.hypot(tx, ty))),
        "mean_displacement": float(np.mean(np.hypot(tx, ty))),
        "jump_recoveries": reasons.count("ok_after_jump_recovery"),
        "fatal_events": reasons.count("fatal_jump_circuit_breaker"),
    })

drift_df = pd.DataFrame(drift_rows)
display(drift_df)
if not drift_df.empty:
    drift_df.to_csv(RESULTS_DIR / "registration_drift_summary.csv", index=False)


## 8. Future Quantification Result Loaders


In [ ]:
EXPECTED_RESULT_FILES = {
    "signal_intensity": RESULTS_DIR / "signal_intensity_summary.csv",
    "snr": RESULTS_DIR / "snr_summary.csv",
    "morphology_3d": RESULTS_DIR / "particle_object_morphology_3d_summary.csv",
    "colocalization": RESULTS_DIR / "reference_target_colocalization_summary.csv",
}

loaded_results = {}
for name, path in EXPECTED_RESULT_FILES.items():
    if path.exists():
        loaded_results[name] = pd.read_csv(path)
        print(f"[LOAD] {name}: {path}")
        display(loaded_results[name].head())
    else:
        print(f"[PENDING] {name}: {path}")
